# DataMate — chạy eval trên Colab (T4 16 GB)
Runtime → *Change runtime type* → **T4 GPU**. Chạy lần lượt từng ô.
Ô 1 gắn Drive, ô 2 lấy code + dữ liệu, ô 3 cài đặt, ô 4 bật vLLM, ô 5–7 chạy eval, ô 8 lưu kết quả.

**Chưa kiểm chứng trên Colab thật**: nếu ô 4 báo lỗi, gửi lại 40 dòng cuối của `/content/vllm.log`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Code và dữ liệu
Cách A (khuyên dùng): nén repo thành `datamate.zip` (không gồm `train/`, `.venv/`, `cache/`) và dữ liệu train thành `train.zip` rồi đặt trong Drive.
Cách B: `git clone` repo riêng của bạn.

In [ ]:
import os, subprocess
DRIVE = '/content/drive/MyDrive/zalo'          # sửa cho đúng thư mục của bạn
REPO  = '/content/datamate'
if not os.path.exists(REPO):
    os.makedirs(REPO, exist_ok=True)
    subprocess.run(['git','clone','https://github.com/DucAn24/zalo_ai.git',REPO], check=True)
    # subprocess.run(['git','clone','<URL_REPO>',REPO], check=True)   # cách B
if not os.path.exists('/content/train'):
    subprocess.run(['unzip','-q','-o',f'{DRIVE}/train.zip','-d','/content/train'], check=True)
os.chdir(REPO)
# thư mục split phải chứa questions.jsonl, answers.jsonl, rulebook.md, calendar.csv, data/
SPLIT = next(r for r,_,f in os.walk('/content/train') if 'questions.jsonl' in f)
print('SPLIT =', SPLIT)

## 3. Cài đặt (hai môi trường Python 3.12 riêng)
Colab đang dùng Python 3.13 mà `vllm==0.8.5.post1` chỉ hỗ trợ <3.13, nên cài vLLM vào venv 3.12 (qua `uv`).
Phần eval cũng chạy trong venv 3.12 riêng với đúng phiên bản của `requirements.txt`. Mất khoảng 5–8 phút lần đầu.

In [ ]:
!pip install -q uv
!uv venv --quiet --python 3.12 /content/vllm_env
!uv pip install --quiet --python /content/vllm_env/bin/python vllm==0.8.5.post1 transformers==4.51.3
!uv venv --quiet --python 3.12 /content/app_env
!uv pip install --quiet --python /content/app_env/bin/python -r requirements.txt

## 4. Bật vLLM (nền) rồi chờ sẵn sàng
Đổi `MODEL` để so sánh: `Qwen/Qwen3-4B-AWQ`, `Qwen/Qwen3-8B-AWQ`, `Qwen/Qwen3-14B-AWQ` (14B cần giảm `MAX_LEN` xuống ~12288).

In [ ]:
import os, subprocess, time, urllib.request
os.environ.update(MODEL='Qwen/Qwen3-8B-AWQ', MAX_LEN='16384', GPU_UTIL='0.90', DTYPE='half', MAX_SEQS='12',
                  PYTHON='/content/vllm_env/bin/python')
log = open('/content/vllm.log', 'w')
srv = subprocess.Popen(['bash', 'scripts/start_vllm.sh'], stdout=log, stderr=subprocess.STDOUT)
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3); print('server ready after', i*5, 's'); break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/vllm.log').read()[-3000:])

In [ ]:
# đưa cấu hình LLM cho pipeline
APP = '/content/app_env/bin/python'
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:8000/v1'
os.environ['LLM_MODEL'] = os.environ['MODEL']
os.environ['WORKERS'] = '8'          # số câu hỏi chạy song song (tăng nếu GPU còn rảnh)
os.environ['PA_PENALTY'] = '1'       # hệ số phạt câu sai; đổi khi biết công thức PA

## 5. Eval nhanh trên nhóm `dev` (100 câu, chia đều theo business)
Chỉ dùng nhóm `dev` để phân tích lỗi và chỉnh prompt.

In [ ]:
!$APP eval.py --split $SPLIT --run --out runs/dev100 --group dev --sample 100

## 6. Nhóm kiểm tra: shop mới (`val_business`) và loại hình lạ (`val_type`)
Không dùng hai nhóm này để chỉnh prompt/luật. Chỉ ghi lại điểm.

In [ ]:
!$APP eval.py --split $SPLIT --run --out runs/val_business --group val_business --sample 60
!$APP eval.py --split $SPLIT --run --out runs/val_type --group val_type --sample 100

## 7. Xem các câu sai của `dev`

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 120, 'display.width', 250)
d = pd.read_csv('runs/dev100/eval_details.csv')
print(d.groupby(['category','result']).size().unstack(fill_value=0))
d[d.result=='wrong'][['business','question','gold','pred','error','sql']].head(25)

## 8. Lưu kết quả về Drive

In [ ]:
!mkdir -p "$DRIVE/runs" && cp -r runs/* "$DRIVE/runs/" && echo saved